# CEOS-style Canonical Identifiers Demo (Simulated)

This notebook demonstrates how a small, simulated “CEOS DB–like” dataset can be used to generate **persistent, harmonised identifiers** for agencies, missions, platforms, instruments, and sensors.

It implements four “incarnations” of names (per Peter Strobl’s proposal):
- **fullName** — human friendly, up to ~256 chars  
- **shortName** — ASCII, ≤32 chars  
- **acronym** — ASCII, ≤16 chars  
- **mnemonic** — file/URI safe, ≤8 chars

It also shows **hierarchical / modular IDs** (agency, mission, platform, instrument), collision handling to enforce uniqueness, and a simple extension to instrument **bands**.


In [1]:
# Bootstrap: ensure required packages are installed (works in Cursor + Colab)
try:
    import pandas as pd  # noqa: F401
except Exception:
    %pip install -q pandas
    import pandas as pd

import re
from typing import Dict, List


In [2]:
# ---------- Import local idkit library ----------

import os, sys, pathlib

# Add this notebook’s directory to sys.path so Python can find idkit.py
nb_dir = pathlib.Path().resolve()
if str(nb_dir) not in sys.path:
    sys.path.insert(0, str(nb_dir))

from idkit import IdKit, IdConfig, add_band_ids, validate_lengths

# Configure and instantiate the IdKit generator
cfg = IdConfig(
    short_max=32,
    acronym_max=16,
    mnemonic_max=8,
    flat_max=48,
    hier_max=64,
    include_platform_in_mnemonic=True,
)
idkit = IdKit(cfg)


In [3]:
# -------------------------------------------------------------------
# Simulated CEOS Database records
#
# In production, this dataset would be pulled directly from:
#   - a CEOS DB API endpoint (JSON → DataFrame), OR
#   - a SQL query (e.g. SELECT agency, mission, platform, instrument ...)
#
# For this demo, we hardcode a minimal representative sample.
# The format mirrors what we expect from the DB so that this
# cell can be swapped out later with a live query.
# -------------------------------------------------------------------

rows = [
    # ESA / Sentinel family
    {"agency":"ESA",  "mission":"Sentinel-1", "platform_code":"A", "instrument":"C-SAR"},
    {"agency":"ESA",  "mission":"Sentinel-1", "platform_code":"B", "instrument":"C-SAR"},
    {"agency":"ESA",  "mission":"Sentinel-1", "platform_code":"C", "instrument":"C-SAR"},
    {"agency":"ESA",  "mission":"Sentinel-2", "platform_code":"A", "instrument":"MSI"},
    {"agency":"ESA",  "mission":"Sentinel-2", "platform_code":"B", "instrument":"MSI"},
    {"agency":"ESA",  "mission":"Sentinel-3", "platform_code":"A", "instrument":"OLCI"},
    {"agency":"ESA",  "mission":"Sentinel-3", "platform_code":"B", "instrument":"OLCI"},

    # USGS / Landsat
    {"agency":"USGS", "mission":"Landsat-8",  "platform_code":"8", "instrument":"OLI"},
    {"agency":"USGS", "mission":"Landsat-9",  "platform_code":"9", "instrument":"OLI"},

    # JAXA / ALOS-2
    {"agency":"JAXA", "mission":"ALOS",     "platform_code":"2",  "instrument":"PALSAR-2"},
]

df = pd.DataFrame(rows)
df


,agency,mission,platform_code,instrument
0,ESA,Sentinel-1,A,C-SAR
1,ESA,Sentinel-1,B,C-SAR
2,ESA,Sentinel-1,C,C-SAR
3,ESA,Sentinel-2,A,MSI
4,ESA,Sentinel-2,B,MSI
5,ESA,Sentinel-3,A,OLCI
6,ESA,Sentinel-3,B,OLCI
7,USGS,Landsat-8,8,OLI
8,USGS,Landsat-9,9,OLI
9,JAXA,ALOS,2,PALSAR-2


In [4]:
# ---------- Mint identifiers for each row (via idkit lib) ----------

# Use the IdKit class we imported earlier to generate IDs for the full dataset
df_ids = idkit.mint_dataframe(df)

# Quick invariant check on length constraints
validation = validate_lengths(df_ids)
print("Primary ID length constraints:")
display(validation)


display_cols = [
    "agency","mission","platform_code","instrument",
    "fullName","shortName","acronym","mnemonic","flat_id","hierarchical_id"
]
df_ids[display_cols]


Primary ID length constraints:


flat_ok     True
hier_ok     True
short_ok    True
acro_ok     True
mnem_ok     True
dtype: bool

,agency,mission,platform_code,instrument,fullName,shortName,acronym,mnemonic,flat_id,hierarchical_id
0,ESA,Sentinel-1,A,C-SAR,ESA Sentinel-1 A C-SAR,Sentinel-1-C-SAR-A,S1_CS_A,SENTCSAA,ESA-SENT1-CSAR-A,A_ESA-M_SENT1-P_A-I_CSAR
1,ESA,Sentinel-1,B,C-SAR,ESA Sentinel-1 B C-SAR,Sentinel-1-C-SAR-B,S1_CS_B,SENTCSAB,ESA-SENT1-CSAR-B,A_ESA-M_SENT1-P_B-I_CSAR
2,ESA,Sentinel-1,C,C-SAR,ESA Sentinel-1 C C-SAR,Sentinel-1-C-SAR-C,S1_CS_C,SENTCSAC,ESA-SENT1-CSAR-C,A_ESA-M_SENT1-P_C-I_CSAR
3,ESA,Sentinel-2,A,MSI,ESA Sentinel-2 A MSI,Sentinel-2-MSI-A,S2_M_A,SENTMSIA,ESA-SENT2-MSI-A,A_ESA-M_SENT2-P_A-I_MSI
4,ESA,Sentinel-2,B,MSI,ESA Sentinel-2 B MSI,Sentinel-2-MSI-B,S2_M_B,SENTMSIB,ESA-SENT2-MSI-B,A_ESA-M_SENT2-P_B-I_MSI
5,ESA,Sentinel-3,A,OLCI,ESA Sentinel-3 A OLCI,Sentinel-3-OLCI-A,S3_O_A,SENTOLCA,ESA-SENT3-OLCI-A,A_ESA-M_SENT3-P_A-I_OLCI
6,ESA,Sentinel-3,B,OLCI,ESA Sentinel-3 B OLCI,Sentinel-3-OLCI-B,S3_O_B,SENTOLCB,ESA-SENT3-OLCI-B,A_ESA-M_SENT3-P_B-I_OLCI
7,USGS,Landsat-8,8,OLI,USGS Landsat-8 8 OLI,Landsat-8-OLI-8,L8_O_8,LANDOLI8,USGS-LANDSAT8-OLI-8,A_USGS-M_LANDSAT8-P_8-I_OLI
8,USGS,Landsat-9,9,OLI,USGS Landsat-9 9 OLI,Landsat-9-OLI-9,L9_O_9,LANDOLI9,USGS-LANDSAT9-OLI-9,A_USGS-M_LANDSAT9-P_9-I_OLI
9,JAXA,ALOS,2,PALSAR-2,JAXA ALOS 2 PALSAR-2,ALOS-PALSAR-2-2,A_P2_2,ALOSPAL2,JAXA-ALOS-PALSAR2-2,A_JAXA-M_ALOS-P_2-I_PALSAR2


In [5]:
# Preview the minted identifiers
display_cols = [
    "agency", "mission", "platform_code", "instrument",
    "fullName", "shortName", "acronym", "mnemonic",
    "flat_id", "hierarchical_id"
]
df_ids[display_cols]


,agency,mission,platform_code,instrument,fullName,shortName,acronym,mnemonic,flat_id,hierarchical_id
0,ESA,Sentinel-1,A,C-SAR,ESA Sentinel-1 A C-SAR,Sentinel-1-C-SAR-A,S1_CS_A,SENTCSAA,ESA-SENT1-CSAR-A,A_ESA-M_SENT1-P_A-I_CSAR
1,ESA,Sentinel-1,B,C-SAR,ESA Sentinel-1 B C-SAR,Sentinel-1-C-SAR-B,S1_CS_B,SENTCSAB,ESA-SENT1-CSAR-B,A_ESA-M_SENT1-P_B-I_CSAR
2,ESA,Sentinel-1,C,C-SAR,ESA Sentinel-1 C C-SAR,Sentinel-1-C-SAR-C,S1_CS_C,SENTCSAC,ESA-SENT1-CSAR-C,A_ESA-M_SENT1-P_C-I_CSAR
3,ESA,Sentinel-2,A,MSI,ESA Sentinel-2 A MSI,Sentinel-2-MSI-A,S2_M_A,SENTMSIA,ESA-SENT2-MSI-A,A_ESA-M_SENT2-P_A-I_MSI
4,ESA,Sentinel-2,B,MSI,ESA Sentinel-2 B MSI,Sentinel-2-MSI-B,S2_M_B,SENTMSIB,ESA-SENT2-MSI-B,A_ESA-M_SENT2-P_B-I_MSI
5,ESA,Sentinel-3,A,OLCI,ESA Sentinel-3 A OLCI,Sentinel-3-OLCI-A,S3_O_A,SENTOLCA,ESA-SENT3-OLCI-A,A_ESA-M_SENT3-P_A-I_OLCI
6,ESA,Sentinel-3,B,OLCI,ESA Sentinel-3 B OLCI,Sentinel-3-OLCI-B,S3_O_B,SENTOLCB,ESA-SENT3-OLCI-B,A_ESA-M_SENT3-P_B-I_OLCI
7,USGS,Landsat-8,8,OLI,USGS Landsat-8 8 OLI,Landsat-8-OLI-8,L8_O_8,LANDOLI8,USGS-LANDSAT8-OLI-8,A_USGS-M_LANDSAT8-P_8-I_OLI
8,USGS,Landsat-9,9,OLI,USGS Landsat-9 9 OLI,Landsat-9-OLI-9,L9_O_9,LANDOLI9,USGS-LANDSAT9-OLI-9,A_USGS-M_LANDSAT9-P_9-I_OLI
9,JAXA,ALOS,2,PALSAR-2,JAXA ALOS 2 PALSAR-2,ALOS-PALSAR-2-2,A_P2_2,ALOSPAL2,JAXA-ALOS-PALSAR2-2,A_JAXA-M_ALOS-P_2-I_PALSAR2


In [6]:
# Demonstrate uniqueness handling with deliberate collisions using idkit
test = pd.DataFrame([
    {"agency":"TEST", "mission":"Demo-1", "platform_code":"A", "instrument":"XCAM"},
    {"agency":"TEST", "mission":"Demo-1", "platform_code":"B", "instrument":"XCAM"},
])

test_ids = idkit.mint_dataframe(test)
test_ids[
    ["agency","mission","platform_code","instrument",
     "fullName","shortName","acronym","mnemonic","flat_id","hierarchical_id"]
]


,agency,mission,platform_code,instrument,fullName,shortName,acronym,mnemonic,flat_id,hierarchical_id
0,TEST,Demo-1,A,XCAM,TEST Demo-1 A XCAM,Demo-1-XCAM-A,D1_X_A,DEMOXCAA,TEST-DEMO1-XCAM-A,A_TEST-M_DEMO1-P_A-I_XCAM
1,TEST,Demo-1,B,XCAM,TEST Demo-1 B XCAM,Demo-1-XCAM-B,D1_X_B,DEMOXCAB,TEST-DEMO1-XCAM-B,A_TEST-M_DEMO1-P_B-I_XCAM


In [7]:
# ---------- Bands / modes mapping and expansion ----------

# Minimal illustrative mapping (extend as needed or swap from DB/API later)
INSTRUMENT_BANDS = {
    "MSI": [
        {"band_code":"B01","common":"coastal"},
        {"band_code":"B02","common":"blue"},
        {"band_code":"B03","common":"green"},
        {"band_code":"B04","common":"red"},
        {"band_code":"B08","common":"nir"},
    ],
    "OLI": [
        {"band_code":"B1","common":"coastal"},
        {"band_code":"B2","common":"blue"},
        {"band_code":"B3","common":"green"},
        {"band_code":"B4","common":"red"},
        {"band_code":"B5","common":"nir"},
    ],
    "C-SAR": [
        {"band_code":"CBAND","common":"c_band"},
    ],
    "PALSAR-2": [
        {"band_code":"LBAND","common":"l_band"},
        {"band_code":"HH","common":"hh_pol"},   # illustrative polarisation mode
        {"band_code":"HV","common":"hv_pol"},
    ],
}

# Use the library helper to create band-level IDs from df_ids
df_bands = add_band_ids(df_ids, INSTRUMENT_BANDS)

# Peek at results
df_bands[
    ["agency","mission","platform_code","instrument","band_code","band_common",
     "flat_id_band","hierarchical_id_band"]
].head(20)


,agency,mission,platform_code,instrument,band_code,band_common,flat_id_band,hierarchical_id_band
0,ESA,Sentinel-1,A,C-SAR,CBAND,c_band,ESA-SENT1-CSAR-A-CBAND,A_ESA-M_SENT1-P_A-I_CSAR-B_CBAND
1,ESA,Sentinel-1,B,C-SAR,CBAND,c_band,ESA-SENT1-CSAR-B-CBAND,A_ESA-M_SENT1-P_B-I_CSAR-B_CBAND
2,ESA,Sentinel-1,C,C-SAR,CBAND,c_band,ESA-SENT1-CSAR-C-CBAND,A_ESA-M_SENT1-P_C-I_CSAR-B_CBAND
3,ESA,Sentinel-2,A,MSI,B01,coastal,ESA-SENT2-MSI-A-B01,A_ESA-M_SENT2-P_A-I_MSI-B_B01
4,ESA,Sentinel-2,A,MSI,B02,blue,ESA-SENT2-MSI-A-B02,A_ESA-M_SENT2-P_A-I_MSI-B_B02
5,ESA,Sentinel-2,A,MSI,B03,green,ESA-SENT2-MSI-A-B03,A_ESA-M_SENT2-P_A-I_MSI-B_B03
6,ESA,Sentinel-2,A,MSI,B04,red,ESA-SENT2-MSI-A-B04,A_ESA-M_SENT2-P_A-I_MSI-B_B04
7,ESA,Sentinel-2,A,MSI,B08,nir,ESA-SENT2-MSI-A-B08,A_ESA-M_SENT2-P_A-I_MSI-B_B08
8,ESA,Sentinel-2,B,MSI,B01,coastal,ESA-SENT2-MSI-B-B01,A_ESA-M_SENT2-P_B-I_MSI-B_B01
9,ESA,Sentinel-2,B,MSI,B02,blue,ESA-SENT2-MSI-B-B02,A_ESA-M_SENT2-P_B-I_MSI-B_B02


## Lightweight shares (subset exports)

For quick reviews or email shares, export small, focused slices of the registry—
e.g., just **Sentinel-2** (MSI) and **ALOS-2** (PALSAR-2). These JSON files keep
the same schema as the full table but are easy to skim.


In [8]:
# ---------- Subset exports for lightweight sharing (into /export) ----------

# Ensure /export folder exists (relative to the notebook location)
out_dir = pathlib.Path("export")
out_dir.mkdir(parents=True, exist_ok=True)

# Mission-based subsets
s2 = df_ids[df_ids["mission"].str.contains("Sentinel-2", case=False, na=False)]
alos2 = df_ids[df_ids["mission"] == "ALOS-2"]

# Optional: band-level subsets (if df_bands exists)
s2_bands = df_bands[df_bands["mission"].str.contains("Sentinel-2", case=False, na=False)] if "df_bands" in globals() else pd.DataFrame()
alos2_bands = df_bands[df_bands["mission"] == "ALOS-2"] if "df_bands" in globals() else pd.DataFrame()

# Write JSON exports
s2.to_json(out_dir / "sentinel2_registry.json", orient="records", indent=2)
alos2.to_json(out_dir / "alos2_registry.json", orient="records", indent=2)
if not s2_bands.empty:
    s2_bands.to_json(out_dir / "sentinel2_registry_bands.json", orient="records", indent=2)
if not alos2_bands.empty:
    alos2_bands.to_json(out_dir / "alos2_registry_bands.json", orient="records", indent=2)

# Write CSV exports
s2.to_csv(out_dir / "sentinel2_registry.csv", index=False)
alos2.to_csv(out_dir / "alos2_registry.csv", index=False)
if not s2_bands.empty:
    s2_bands.to_csv(out_dir / "sentinel2_registry_bands.csv", index=False)
if not alos2_bands.empty:
    alos2_bands.to_csv(out_dir / "alos2_registry_bands.csv", index=False)

print("Wrote subset exports into:", out_dir.resolve())
for f in sorted(out_dir.glob("*")):
    print(" -", f.name)


Wrote subset exports into: /Users/georgedyke/GitHub/ceos-db-toolkit/colab-notebooks/canonical_id_demo/export
 - alos2_registry.csv
 - alos2_registry.json
 - alos2_registry_bands.csv
 - alos2_registry_bands.json
 - sentinel2_registry.csv
 - sentinel2_registry.json
 - sentinel2_registry_bands.csv
 - sentinel2_registry_bands.json
